In [4]:
!pip install easyocr transformers openai-whisper gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 12.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a219aa5665cd394eb5b19b2879f63eb1e2dd56d62b49760293bc8ab816e05d1d
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [7]:
import torch
from transformers import pipeline
import easyocr
import whisper
import gradio as gr

# 1. Initialize Models (This will download them the first time)
# OCR for Image-to-Text
reader = easyocr.Reader(['en'])

# Translation (English to Urdu)
# We use a Transformer model for contextual accuracy
translator = pipeline("translation_en_to_ur", model="Helsinki-NLP/opus-mt-en-ur")

# Voice (Transcription and Translation)
voice_model = whisper.load_model("base")

# 2. Define Logic Functions
def translate_text(text):
    if not text.strip(): return ""
    result = translator(text)
    return result[0]['translation_text']

def translate_image(img):
    # OCR to get text
    results = reader.readtext(img)
    extracted_text = " ".join([res[1] for res in results])
    # Translate extracted text
    return translate_text(extracted_text)

def translate_voice(audio_path):
    # Whisper transcribes and can translate to English,
    # but for Urdu we transcribe first then translate
    result = voice_model.transcribe(audio_path)
    return translate_text(result['text'])

# 3. Create the App Interface (using Gradio)
with gr.Blocks() as demo:
    gr.Markdown("# 🇵🇰 AI English to Urdu Translator")

    with gr.Tab("Text Translation"):
        txt_input = gr.Textbox(label="Enter English")
        txt_output = gr.Textbox(label="Urdu Result")
        btn_txt = gr.Button("Translate Text")
        btn_txt.click(translate_text, inputs=txt_input, outputs=txt_output)

    with gr.Tab("Image Translation"):
        img_input = gr.Image(type="filepath", label="Upload Image/Screenshot")
        img_output = gr.Textbox(label="Urdu Result")
        btn_img = gr.Button("Extract & Translate")
        btn_img.click(translate_image, inputs=img_input, outputs=img_output)

    with gr.Tab("Voice Translation"):
        aud_input = gr.Audio(type="filepath", label="Record or Upload Audio")
        aud_output = gr.Textbox(label="Urdu Result")
        btn_aud = gr.Button("Transcribe & Translate")
        btn_aud.click(translate_voice, inputs=aud_input, outputs=aud_output)

demo.launch(debug=True)

SyntaxError: incomplete input (2535253662.py, line 59)